# Video -> wav
Converts videos in `data/videos/` to wav in `data/raw_audios/`. Optional.

The wav is mono, at the video's own sample rate: it is never resampled. The pipeline needs 48 kHz, so a video with other audio (often 44.1 kHz) is reported here, and 01 and 02 stop on it: find a 48 kHz source for it.

In [3]:
import subprocess

import soundfile as sf
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
out_dir = root / "data" / "raw_audios"
out_dir.mkdir(parents=True, exist_ok=True)

videos = sorted(p for p in (root / "data" / "videos").glob("*") if p.suffix.lower() in {".mp4", ".mkv", ".mov", ".webm"})
for video in videos:
    wav = out_dir / f"{video.stem}.wav"
    if wav.exists():
        print("exists:", wav.name, f"{sf.info(str(wav)).samplerate} Hz")
        continue
    subprocess.run(["ffmpeg", "-loglevel", "error", "-i", str(video), "-vn", "-ac", "1", "-c:a", "pcm_s16le", str(wav)], check=True)
    sr = sf.info(str(wav)).samplerate
    print("saved:", wav.name, f"{sr} Hz" + ("" if sr == 48000 else "  <- NOT 48 kHz: 01 and 02 will stop on it"))